# Notebook for creating plots of the grid-search metrics
Needs to be run after `workflow/gmm_clustering.ipynb`.

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path

import pandas as pd
import plotnine as p9
from plotnine.composition import Beside, Wrap, plot_layout

from utils.plotting_utils import make_plot_grid_search, make_plot_parameter_selection, metric_labels
from utils.saving_utils import load_object, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
grid_search_boot = load_object(Path("outputs/gmm_clustering/gmm_boot_discov.pkl"))
model_discov = load_object(Path("outputs/gmm_clustering/model_discov.pkl"))
metric_summaries = grid_search_boot.summaries_.sort_values(["metric", "n_components"]).reset_index(drop=True)
paired_gaps = grid_search_boot.paired_difference_summaries_.sort_values("n_components").reset_index(drop=True)

# Set outputs folder
output_folder = Path("outputs/gmm_clustering")
output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# Set the grid-search parameter
x_param = "n_components"

selected_k = grid_search_boot.best_params_[x_param]
best_k = metric_summaries.loc[lambda x: x["metric"].eq(grid_search_boot.selection_metric)].loc[
    lambda x: x["mean"].idxmax() if grid_search_boot.selection_metric_direction == "max" else x["mean"].idxmin(),
    x_param,
]
eligible_k = sorted(params[x_param] for params in grid_search_boot.eligible_params_)

# Set plot labels
mcse_caption = "Band: estimate ± 1 Monte Carlo standard error; imputation uncertainty is not included."

# Set x-axis scale
x_scale = p9.scale_x_continuous(breaks=[2, 10, 20, 30, 40, 50], expand=(0.02, 0))

In [ ]:
# Make plots of parameter selection
selection_metric_plot, differences_plot, tie_break_plot = make_plot_parameter_selection(
    grid_search_boot, x_param, errorbars_var=None, ribbon_var="mcse"
)
for plot in [selection_metric_plot, differences_plot, tie_break_plot]:
    if plot is not None:
        plot += p9.labs(y="")
        plot += x_scale
selection_metric_label = metric_labels.get(grid_search_boot.selection_metric, grid_search_boot.selection_metric)
selection_metric_plot += p9.labs(title=selection_metric_label)
differences_plot += p9.labs(title=f"{selection_metric_label} gap relative to K = {best_k}")
if tie_break_plot is not None:
    tie_break_plot += p9.labs(title=metric_labels.get(grid_search_boot.tie_break, grid_search_boot.tie_break))

# Combine plots
parameter_selection_plot = selection_metric_plot | differences_plot + p9.theme(legend_position="none")
if tie_break_plot is not None:
    parameter_selection_plot |= tie_break_plot + p9.theme(legend_position="none")
parameter_selection_plot &= p9.theme(figure_size=(13.5, 5.5), dpi=150)
parameter_selection_plot.show()

# Save plots
save_plot(parameter_selection_plot, output_folder / "parameter_selection_plot.png", dpi=150)

In [ ]:
# Make grid-search plots for each metric
metric_grid_search_plots = {}
for metric in metric_summaries["metric"].drop_duplicates():
    plot = make_plot_grid_search(
        metric_summaries,
        x_param=x_param,
        metrics=metric,
        selected_value=selected_k,
        errorbars_var=None,
        ribbon_var="mcse",
    )
    plot += p9.labs(title=metric_labels.get(metric, metric), y="")
    plot += x_scale
    if metric in ["ari", "assoc", "nmi"]:
        plot += p9.coord_cartesian(ylim=(0, 1))
    elif metric == "si":
        plot += p9.coord_cartesian(ylim=(0, 0.65))
    metric_grid_search_plots[metric] = plot

    # Save individual metric plots
    save_plot(
        plot + p9.labs(caption=mcse_caption) + p9.theme(figure_size=(11.5, 5.8)),
        output_folder / f"plot_{metric}_grid_search.png",
        dpi=150,
    )

# Combine the main diagnostic plots
plot_metrics = ["si", "dbi", "ari"]
main_metrics_grid_search_plot = Beside([metric_grid_search_plots[metric] for metric in plot_metrics])

main_metrics_grid_search_plot &= p9.theme(figure_size=(15, 5.5), dpi=150)
main_metrics_grid_search_plot.show()

# Combine the backup metric plots
metrics_grid_search_backup = (
    Wrap([plot for metric, plot in metric_grid_search_plots.items() if metric not in plot_metrics])
    + plot_layout(ncol=3)
)
metrics_grid_search_backup &= p9.theme(figure_size=(16, 9), dpi=150)
metrics_grid_search_backup.show()

# Save plots
save_plot(main_metrics_grid_search_plot, output_folder / "main_metrics_grid_search_plot.png", dpi=150)
save_plot(metrics_grid_search_backup, output_folder / "plot_metrics_grid_search_backup.png", dpi=150)